# Coastal Classification Using GeoAI Python Package

This notebook is based on the template provided by GeoAI "Train a Semantic Segmentation Model using Segmentation-Models-PyTorch"

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/train_segmentation_model.ipynb)

It demonstrates how to train a semantic segmentation model for coastal ridge detection using the [segmentation-models-pytorch](https://smp.readthedocs.io) library. This approach treats the task as pixel-level binary classification.

## Install packages



In [ ]:
pip install geoai-py==0.28.0 # Installation of GeoAI python package 0.28.0

## Import training data and Libraries



In [ ]:
import geoai

from google.colab import drive
import os


drive.mount('/content/drive') # This project was done using colab, therefore google drive was used for filepaths

proj_dir = '/content/drive/MyDrive/aiproj' # Project directory

## Train semantic segmentation model

Now we'll train a semantic segmentation model using the new `train_segmentation_model` function. This function supports various architectures from `segmentation-models-pytorch`:

- **Architectures**: `unet`, `unetplusplus` `deeplabv3`, `deeplabv3plus`, `fpn`, `pspnet`, `linknet`, `manet`
- **Encoders**: `resnet34`, `resnet50`, `efficientnet-b0`, `mobilenet_v2`, etc.

For more details, please refer to the [segmentation-models-pytorch documentation](https://smp.readthedocs.io/en/latest/models.html).

### Example 1: U-Net with ResNet34 encoder


In [ ]:
geoai.train_segmentation_model(
    images_dir="/content/drive/MyDrive/aiproj/images", # Image tiles
    labels_dir="/content/drive/MyDrive/aiproj/data/linear_buffered_masks", # Label tiles
    output_dir="/content/drive/MyDrive/aiproj/model3", # Output directory for model training history and best performing model
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=3, # The channels were changed depending on how many bands were used
    num_classes=2,  # Background and ridge (class 1 and 0)
    batch_size=8,
    num_epochs=30, # Chosen due to computational limitations
    learning_rate=0.0003,
    val_split=0.2, # Splits the image/label data into training and validation
    verbose=True,
)

## Run inference

Now we'll use the trained model to make predictions on the test image.

In [ ]:
# Define paths
masks_path = "/content/drive/MyDrive/aiproj/data/linear_buffered_masks"
model_path = "/content/drive/MyDrive/aiproj/model/best_model.pth"
# Use a test raster for inference
test_raster_path = '/content/drive/MyDrive/aiproj/test_images/image_9.tif'

In [ ]:
predicted_mask_path = os.path.join(masks_path, 'predicted_mask.tif')

# Run semantic segmentation inference
geoai.semantic_segmentation(
    input_path=test_raster_path,
    output_path=predicted_mask_path,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=526,
    overlap=256,
    batch_size=4,
)

In [ ]:
# Run inference with probability output
classified_mask_with_prob_path = os.path.join(masks_path, 'classified_mask_with_prob.tif')
probability_output_file = os.path.join(masks_path, "probability_mask.tif")

geoai.semantic_segmentation(
    input_path=test_raster_path,
    output_path=classified_mask_with_prob_path,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=526,
    overlap=256,
    batch_size=4,
    probability_path=probability_output_file,  # Output probability map file
)

You can also control the classification threshold for binary segmentation. By default, argmax is used, but you can specify a custom threshold:

In [ ]:
# Run inference with custom probability threshold
masks_path_threshold = "naip_test_semantic_prediction_threshold2.tif"

geoai.semantic_segmentation(
    input_path=test_raster_path,
    output_path=masks_path_threshold,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=4,
    probability_threshold=0.3,  # Only classify as ridge if probability >= 0.7
)

## Model Performance Analysis

Let's examine the training curves and model performance:

In [ ]:
from geoai.utils import plot_performance_metrics
hist = "/content/drive/MyDrive/aiproj/model3/training_history.pth"
df = plot_performance_metrics(hist)


## Performance Metrics

**IoU (Intersection over Union)** and **Dice score** are both popular metrics used to evaluate the similarity between two binary masks—often in image segmentation tasks. While they are related, they are not the same.

---

### 🔸 **Definitions**

#### **IoU (Jaccard Index)**

$$
\text{IoU} = \frac{|A \cap B|}{|A \cup B|}
$$

* Measures the overlap between predicted region $A$ and ground truth region $B$ relative to their union.
* Ranges from 0 (no overlap) to 1 (perfect overlap).

#### **Dice Score (F1 Score for Sets)**

$$
\text{Dice} = \frac{2|A \cap B|}{|A| + |B|}
$$

* Measures the overlap between $A$ and $B$, but gives more weight to the intersection.
* Also ranges from 0 to 1.

---

### 🔸 **Key Differences**

| Metric   | Formula                     | Penalizes                      | Sensitivity                      |
| -------- | --------------------------- | ------------------------------ | -------------------------------- |
| **IoU**  | $\frac{TP}{TP + FP + FN}$   | FP and FN equally              | Less sensitive to small objects  |
| **Dice** | $\frac{2TP}{2TP + FP + FN}$ | Less harsh on small mismatches | More sensitive to small overlaps |

> TP: True Positive, FP: False Positive, FN: False Negative

---

### 🔸 **Relationship**

Dice and IoU are mathematically related:

$$
\text{Dice} = \frac{2 \cdot \text{IoU}}{1 + \text{IoU}} \quad \text{or} \quad \text{IoU} = \frac{\text{Dice}}{2 - \text{Dice}}
$$

---

### 🔸 **When to Use What**

* **IoU**: Common in object detection and semantic segmentation benchmarks (e.g., COCO, Pascal VOC).
* **Dice**: Preferred in medical imaging and when class imbalance is an issue, due to its sensitivity to small regions.